In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

# -------------------------------
# Load and merge data
# -------------------------------
baseline_features = pd.read_csv('data/processed/instacart/features/baseline_features.csv')
labels = pd.read_csv('data/processed/instacart/labels/instacart_phase2_decay_labels.csv')

model_data = baseline_features.merge(
    labels[['user_id', 'order_id', 'early_decay_label', 'split']],
    on=['user_id', 'order_id'], how='inner'
)

# -------------------------------
# Split
# -------------------------------
train = model_data[model_data['split'] == 'train'].copy()
val = model_data[model_data['split'] == 'validation'].copy()
test = model_data[model_data['split'] == 'test'].copy()

feature_cols = [
    'base_user_tenure_days',
    'base_total_orders_to_date',
    'base_avg_days_between_orders',
    'base_avg_basket_size_to_date',
    'base_avg_reorder_ratio_to_date',
    'base_order_dow',
    'base_order_hour'
]

X_train, y_train = train[feature_cols], train['early_decay_label']
X_val, y_val = val[feature_cols], val['early_decay_label']
X_test, y_test = test[feature_cols], test['early_decay_label']

# -------------------------------
# Train Logistic Regression (needs scaled features)
# -------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg.fit(X_train_scaled, y_train)

# -------------------------------
# Train XGBoost (raw features, no scaling needed)
# -------------------------------
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42
)
xgb_model.fit(X_train, y_train)

# -------------------------------
# Evaluate both on validation set
# -------------------------------
def evaluate(name, y_true, probs, base_rate):
    preds = (probs >= 0.5).astype(int)
    results = pd.DataFrame({'y_true': y_true.values, 'risk_score': probs}).sort_values('risk_score', ascending=False)
    top5 = results.head(int(len(results) * 0.05))['y_true'].mean()
    top10 = results.head(int(len(results) * 0.10))['y_true'].mean()

    print(f"\n===== {name} =====")
    print(f"ROC-AUC: {roc_auc_score(y_true, probs):.4f}")
    print(f"PR-AUC: {average_precision_score(y_true, probs):.4f}")
    print(f"Top 5% lift: {top5/base_rate:.2f}x | Top 10% lift: {top10/base_rate:.2f}x")
    print(classification_report(y_true, preds))

base_rate = y_val.mean()
val_probs_logreg = logreg.predict_proba(X_val_scaled)[:, 1]
val_probs_xgb = xgb_model.predict_proba(X_val)[:, 1]

evaluate("Logistic Regression (baseline features)", y_val, val_probs_logreg, base_rate)
evaluate("XGBoost (baseline features)", y_val, val_probs_xgb, base_rate)

# -------------------------------
# Save models and test-set predictions
# -------------------------------
os.makedirs('models', exist_ok=True)
os.makedirs('data/processed/instacart/predictions', exist_ok=True)

joblib.dump(logreg, 'models/logreg_baseline.pkl')
joblib.dump(xgb_model, 'models/xgboost_baseline.pkl')

predictions = test[['user_id', 'order_id', 'early_decay_label']].copy()
predictions['risk_score_logreg'] = logreg.predict_proba(X_test_scaled)[:, 1]
predictions['risk_score_xgboost'] = xgb_model.predict_proba(X_test)[:, 1]
predictions.to_csv('data/processed/instacart/predictions/baseline_model_predictions.csv', index=False)

print(f"\nSaved {len(predictions):,} test-set predictions")
print("Saved models: models/logreg_baseline.pkl, models/xgboost_baseline.pkl")


===== Logistic Regression (baseline features) =====
ROC-AUC: 0.6189
PR-AUC: 0.2433
Top 5% lift: 1.83x | Top 10% lift: 1.65x
              precision    recall  f1-score   support

           0       0.87      0.57      0.69    322829
           1       0.22      0.58      0.32     67238

    accuracy                           0.57    390067
   macro avg       0.54      0.58      0.50    390067
weighted avg       0.76      0.57      0.63    390067


===== XGBoost (baseline features) =====
ROC-AUC: 0.6603
PR-AUC: 0.2692
Top 5% lift: 1.98x | Top 10% lift: 1.80x
              precision    recall  f1-score   support

           0       0.90      0.49      0.63    322829
           1       0.23      0.74      0.35     67238

    accuracy                           0.53    390067
   macro avg       0.56      0.61      0.49    390067
weighted avg       0.78      0.53      0.58    390067


Saved 388,037 test-set predictions
Saved models: models/logreg_baseline.pkl, models/xgboost_baseline.pkl
